# Notebook 12 — Distributional Views of the Private–Government Gap

The headline arithmetic gap (mean ≈ 0.037 after quality filtering) obscures a bimodal distribution: the median district gap is **−0.105**, meaning more than half of districts show government outperforming private. This notebook provides the visuals that expose the full distribution.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

PROV_COLOURS = {
    'Punjab':           '#1f77b4',
    'Sindh':            '#d62728',
    'KPK':              '#2ca02c',
    'Balochistan':      '#ff7f0e',
    'AJK':              '#9467bd',
    'Gilgit-Baltistan': '#8c564b',
    'ICT':              '#7f7f7f',
}
GOVT_CLR = '#1f77b4'
PVT_CLR  = '#d62728'

print('Setup complete.')

## Load and filter district_summary

In [ ]:
ds_raw = pd.read_csv('../Data:/district_summary.csv')

ds = ds_raw[
    (ds_raw['n_government'] >= 100) &
    (ds_raw['n_private']    >= 50)  &
    (ds_raw['arithmetic_gap'].notna())
].copy()

print(f'Retained {len(ds)} of {len(ds_raw)} districts '
      f'(govt ≥ 100, pvt ≥ 50, gap non-null).')
print()

gap_mean   = ds['arithmetic_gap'].mean()
gap_median = ds['arithmetic_gap'].median()
print(f'Mean gap:   {gap_mean:+.3f}')
print(f'Median gap: {gap_median:+.3f}')
print()
print(f"Gap > +0.1 (pvt clearly better): {(ds['arithmetic_gap'] >  0.1).sum()} "
      f"= {(ds['arithmetic_gap'] >  0.1).mean():.1%}")
print(f"Gap < −0.1 (govt clearly better): {(ds['arithmetic_gap'] < -0.1).sum()} "
      f"= {(ds['arithmetic_gap'] < -0.1).mean():.1%}")
print(f"|gap| ≤ 0.1 (tied):               {(ds['arithmetic_gap'].abs() <= 0.1).sum()} "
      f"= {(ds['arithmetic_gap'].abs() <= 0.1).mean():.1%}")

## Figure 12a — Violin plot of arithmetic_gap by province

In [ ]:
# Province order by median gap (highest private advantage first)
prov_order = (
    ds.groupby('province')['arithmetic_gap']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

# ICT has only 1 district after filtering — violin requires ≥2 points;
# render it as a scatter point instead
violin_provs  = [p for p in prov_order if ds[ds['province'] == p].shape[0] >= 2]
scatter_provs = [p for p in prov_order if p not in violin_provs]

groups = [ds.loc[ds['province'] == p, 'arithmetic_gap'].values for p in violin_provs]
labels = [f"{p}\n(n={len(g)})" for p, g in zip(violin_provs, groups)]

fig, ax = plt.subplots(figsize=(10, 5))

parts = ax.violinplot(groups, positions=range(1, len(groups) + 1),
                      showmedians=False, showextrema=False)

for i, (body, prov) in enumerate(zip(parts['bodies'], violin_provs), start=1):
    body.set_facecolor(PROV_COLOURS.get(prov, '#aec7e8'))
    body.set_alpha(0.55)
    body.set_edgecolor('grey')
    body.set_linewidth(0.6)

    g = ds.loc[ds['province'] == prov, 'arithmetic_gap']
    q1, med, q3 = g.quantile([0.25, 0.5, 0.75])
    ax.vlines(i, q1, q3, color='black', linewidth=4, alpha=0.6, zorder=3)
    ax.scatter(i, med, color='white', s=35, zorder=4, edgecolors='black', linewidths=0.8)

# Single-district provinces as scatter points
for prov in scatter_provs:
    g = ds.loc[ds['province'] == prov, 'arithmetic_gap'].values
    x_pos = len(violin_provs) + 1 + scatter_provs.index(prov)
    ax.scatter([x_pos] * len(g), g,
               color=PROV_COLOURS.get(prov, 'grey'),
               s=60, zorder=4, marker='D', edgecolors='black', linewidths=0.5)
    labels.append(f"{prov}\n(n={len(g)})")

all_positions = list(range(1, len(violin_provs) + len(scatter_provs) + 1))
ax.set_xticks(all_positions)
ax.set_xticklabels(labels, fontsize=8)

ax.axhline(0, color='black', lw=1.0, ls='-', alpha=0.7, zorder=2)
ax.axhline(gap_mean,   color='steelblue', lw=1.2, ls='--', alpha=0.8, label=f'National mean ({gap_mean:+.3f})')
ax.axhline(gap_median, color='firebrick',  lw=1.2, ls=':',  alpha=0.8, label=f'National median ({gap_median:+.3f})')

ax.set_ylabel('Arithmetic gap (private − government mean score)')
ax.set_title('Distribution of District-Level Private–Government Arithmetic Gaps by Province\n'
             '(districts with govt ≥ 100 and pvt ≥ 50 children; box = IQR; white dot = median)')
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(-1.8, 2.7)

plt.tight_layout()
plt.savefig('../outputs/figures/12a_gap_violin_by_province.png', dpi=150, bbox_inches='tight')
plt.savefig('../outputs/figures/12a_gap_violin_by_province.pdf', bbox_inches='tight')
plt.show()
print('Saved 12a_gap_violin_by_province.')

## Figure 12b — Histogram of all district-level gaps

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(ds['arithmetic_gap'], bins=30, color='#aec7e8', edgecolor='white',
        linewidth=0.5, alpha=0.9, zorder=2)

ax.axvline(0, color='black', lw=1.0, ls='-', alpha=0.5, zorder=3)

ax.axvline(gap_mean, color='steelblue', lw=2.0, ls='--', zorder=4)
ax.text(gap_mean + 0.03, ax.get_ylim()[1] * 0.93,
        f'Mean\n{gap_mean:+.3f}',
        color='steelblue', fontsize=8, va='top')

ax.axvline(gap_median, color='firebrick', lw=2.0, ls=':', zorder=4)
ax.text(gap_median - 0.04, ax.get_ylim()[1] * 0.93,
        f'Median\n{gap_median:+.3f}',
        color='firebrick', fontsize=8, va='top', ha='right')

# Shade regions
ax.axvspan(ds['arithmetic_gap'].min() - 0.1, -0.1,
           alpha=0.06, color=GOVT_CLR, zorder=1)
ax.axvspan(0.1, ds['arithmetic_gap'].max() + 0.1,
           alpha=0.06, color=PVT_CLR,  zorder=1)

ax.text(-0.9, ax.get_ylim()[1] * 0.7, 'Govt better\n(gap < −0.1)',
        color=GOVT_CLR, fontsize=8, alpha=0.9)
ax.text(0.6, ax.get_ylim()[1] * 0.7, 'Pvt better\n(gap > +0.1)',
        color=PVT_CLR, fontsize=8, alpha=0.9)

ax.set_xlabel('Arithmetic gap (private − government mean score)')
ax.set_ylabel('Number of districts')
ax.set_title(f'Histogram of District-Level Arithmetic Gaps (n = {len(ds)} filtered districts)')
plt.tight_layout()
plt.savefig('../outputs/figures/12b_gap_histogram.png', dpi=150, bbox_inches='tight')
plt.savefig('../outputs/figures/12b_gap_histogram.pdf', bbox_inches='tight')
plt.show()
print('Saved 12b_gap_histogram.')

## Table — Province breakdown: pvt better / govt better / tied

In [ ]:
def province_row(g):
    n = len(g)
    return pd.Series({
        'n_districts':      n,
        'median_gap':       round(g['arithmetic_gap'].median(), 3),
        'mean_gap':         round(g['arithmetic_gap'].mean(),   3),
        'pct_pvt_better':   round((g['arithmetic_gap'] >  0.1).sum() / n * 100, 1),
        'pct_tied':         round((g['arithmetic_gap'].abs() <= 0.1).sum() / n * 100, 1),
        'pct_govt_better':  round((g['arithmetic_gap'] < -0.1).sum() / n * 100, 1),
    })

prov_tbl = (
    ds.groupby('province', group_keys=False)
    .apply(province_row)
    .reset_index()
    .sort_values('median_gap', ascending=False)
)

# National summary row
nat_row = pd.DataFrame([{
    'province':         'NATIONAL',
    'n_districts':      len(ds),
    'median_gap':       round(gap_median, 3),
    'mean_gap':         round(gap_mean,   3),
    'pct_pvt_better':   round((ds['arithmetic_gap'] >  0.1).mean() * 100, 1),
    'pct_tied':         round((ds['arithmetic_gap'].abs() <= 0.1).mean() * 100, 1),
    'pct_govt_better':  round((ds['arithmetic_gap'] < -0.1).mean() * 100, 1),
}])
prov_tbl = pd.concat([prov_tbl, nat_row], ignore_index=True)

print(prov_tbl.to_string(index=False))

prov_tbl.to_csv('../outputs/distribution_by_province.csv', index=False)
print('\nSaved outputs/distribution_by_province.csv')

## Figure 12c — Child-level score distribution: Government vs Private by province

The bar chart below shows the share of children at each integer arithmetic score (1–7), split by school type, for Punjab, Sindh, KPK, and Balochistan. This reveals **where** in the distribution the private-school advantage is concentrated.

In [ ]:
PLOT_PROVS = ['Punjab', 'Sindh', 'KPK', 'Balochistan']

df = pd.read_csv('../Data:/child_merged.csv',
                 usecols=['province', 'school_type', 'arithmetic_score'])
df = df[
    df['province'].isin(PLOT_PROVS) &
    df['school_type'].isin(['Government', 'Private']) &
    df['arithmetic_score'].notna()
].copy()
df['arithmetic_score'] = df['arithmetic_score'].astype(int)

SCORES = list(range(1, 8))
x = np.arange(len(SCORES))
W = 0.38

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=False)
axes = axes.flatten()

for ax, prov in zip(axes, PLOT_PROVS):
    sub = df[df['province'] == prov]
    gov = sub[sub['school_type'] == 'Government']
    pvt = sub[sub['school_type'] == 'Private']

    n_g = len(gov)
    n_p = len(pvt)

    gov_pct = [gov['arithmetic_score'].eq(s).sum() / n_g * 100 for s in SCORES]
    pvt_pct = [pvt['arithmetic_score'].eq(s).sum() / n_p * 100 for s in SCORES]

    ax.bar(x - W/2, gov_pct, width=W, color=GOVT_CLR, alpha=0.75,
           label=f'Government (n={n_g:,})', edgecolor='white', linewidth=0.3)
    ax.bar(x + W/2, pvt_pct, width=W, color=PVT_CLR,  alpha=0.75,
           label=f'Private (n={n_p:,})',    edgecolor='white', linewidth=0.3)

    ax.set_xticks(x)
    ax.set_xticklabels(SCORES)
    ax.set_xlabel('Arithmetic score (1 = lowest, 7 = highest)', fontsize=8)
    ax.set_ylabel('Share of students (%)', fontsize=8)
    ax.set_title(prov, fontweight='bold')
    ax.legend(fontsize=7, framealpha=0.7)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))

    # Annotate score-7 gap
    g7, p7 = gov_pct[-1], pvt_pct[-1]
    delta7  = p7 - g7
    sign    = '+' if delta7 >= 0 else ''
    ax.text(0.97, 0.96,
            f'Score-7: govt {g7:.1f}%  pvt {p7:.1f}%\n(Δ pvt−govt: {sign}{delta7:.1f} pp)',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey', alpha=0.8))

fig.suptitle('Child-Level Arithmetic Score Distribution: Government vs Private\n'
             '(share of students at each score level; Government=blue, Private=red)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/12c_score_distribution_by_province.png', dpi=150, bbox_inches='tight')
plt.savefig('../outputs/figures/12c_score_distribution_by_province.pdf', bbox_inches='tight')
plt.show()
print('Saved 12c_score_distribution_by_province.')

## Summary: What the distributional view changes

The national-mean framing (mean gap +0.037 after quality filtering) is misleading in two directions at once. First, the **distribution is bimodal**: the median district gap is −0.105 and exactly half (50%) of filtered districts sit in the govt-clearly-better region (gap < −0.1), while only 41% are in the pvt-clearly-better region. A positive mean is consistent with a majority of districts favoring government schools because a handful of districts — particularly in Sindh — have extreme positive outliers (up to +2.25) that pull the national average rightward. Second, the **province-level story is almost the inverse of the headline**: Punjab, with 34 of the 102 filtered districts, has a median gap of −0.275 and 68% of its districts favoring government schools, while Sindh (9 districts, median +0.340) drives most of the private advantage. Third, the child-level score distributions clarify *where* in the distribution these gaps arise: in Sindh, private schools have far fewer students at score-1 (9% vs 18%) and far more at score-7 (28% vs 17%), suggesting an advantage across the full range; in Punjab, the distributions are nearly mirror images and government schools actually lead at score-7 (44% vs 40%), indicating that the Punjab government advantage is concentrated in keeping children out of the lowest performance band. Together, these findings suggest that calling this a national private-school advantage is accurate only for Sindh and parts of KPK; it is factually incorrect for Punjab, which accounts for 33% of filtered districts.